# 04 — Indexação BM25
**Parte 1 — Recuperação Clássica**  
Constrói o índice BM25Okapi sobre o dataset limpo e demonstra buscas.

---

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# Garante que src/ está no path quando rodando do diretório notebooks/
ROOT = Path().resolve().parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from music_search.indexer_raquel import MusicIndex
from music_search.preprocessing_raquel import preprocessar_tokens

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

CORPUS_PATH = ROOT  / 'songs_clean.csv'
INDEX_PATH  = ROOT  / 'bm25_index.pkl'

print('ROOT:', ROOT)
print('Dataset:', CORPUS_PATH.exists())

## 1. Carregar dataset limpo

In [ ]:
df = pd.read_csv(CORPUS_PATH, low_memory=False)
print(f'Linhas: {len(df):,}  |  Colunas: {df.shape[1]}')
df[['track_name', 'artist_name', 'album_name', 'text_field', 'popularity']].head()

## 2. Verificar pipeline de tokenização
Antes de indexar, confirme como o pré-processamento transforma os textos.

In [ ]:
exemplos = df['text_field'].dropna().head(5).tolist()

print('=' * 60)
for texto in exemplos:
    tokens = preprocessar_tokens(texto, use_stemming=True)
    print(f'ENTRADA : {texto}')
    print(f'TOKENS  : {tokens}')
    print('-' * 60)

## 3. Construir e salvar o índice BM25

In [ ]:
%%time
idx = MusicIndex.build(
    df,
    text_col='text_field',
    k1=1.5,
    b=0.75,
    use_stemming=True,
)
print(idx)

In [ ]:
# Estatísticas do índice
stats = idx.stats()
for k, v in stats.items():
    print(f'  {k:<20}: {v}')

In [ ]:
# Salvar em disco
saved = idx.save(INDEX_PATH)
print(f'Índice salvo em: {saved}')

## 4. Carregar índice do disco (simula uso pelos outros membros)

In [ ]:
idx = MusicIndex.load(INDEX_PATH)
print(idx)

## 5. Buscas de exemplo

In [ ]:
def exibir_resultados(query: str, k: int = 10) -> pd.DataFrame:
    """Busca e retorna um DataFrame formatado para análise."""
    results = idx.search(query, k=k)

    if not results:
        print(f'Nenhum resultado para "{query}"')
        return pd.DataFrame()

    rows = [{
        'rank':        r.rank,
        'score':       round(r.score, 4),
        'track_name':  r.track_name,
        'artist_name': r.artist_name,
        'album_name':  r.album_name,
        'popularity':  r.popularity,
        'energy':      r.energy,
        'danceability': r.danceability,
    } for r in results]

    result_df = pd.DataFrame(rows)
    print(f'\nQuery: "{query}"  |  {len(results)} resultados')
    print('=' * 70)
    return result_df

In [ ]:
# Busca por nome de artista
df_r1 = exibir_resultados('The Beatles', k=10)
display(df_r1)

In [ ]:
# Busca por nome de faixa
df_r2 = exibir_resultados('Bohemian Rhapsody', k=10)
display(df_r2)

In [ ]:
# Busca por álbum
df_r3 = exibir_resultados('Dark Side of the Moon', k=10)
display(df_r3)

In [ ]:
# Busca com múltiplos termos
df_r4 = exibir_resultados('folklore', k=10)
display(df_r4)

## 6. Análise de scores — distribuição e decaimento

In [ ]:
# Distribiuição dos scores para uma query
query_analise = 'rock guitar'
scores_all = idx.get_scores_raw(query_analise)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histograma de todos os scores
axes[0].hist(scores_all[scores_all > 0], bins=60,
             color=sns.color_palette('muted')[0], edgecolor='white')
axes[0].set_title(f'Distribuição de scores BM25\nQuery: "{query_analise}"')
axes[0].set_xlabel('Score BM25')
axes[0].set_ylabel('Frequência (docs com score > 0)')

# Decaimento dos top-50 scores
top50_scores = np.sort(scores_all)[::-1][:50]
axes[1].plot(range(1, 51), top50_scores, 'o-',
             color=sns.color_palette('muted')[2], markersize=4)
axes[1].set_title('Decaimento dos top-50 scores')
axes[1].set_xlabel('Rank')
axes[1].set_ylabel('Score BM25')
axes[1].axvline(10, color='red', linestyle='--', alpha=0.5, label='top-10')
axes[1].legend()

plt.suptitle(f'Análise de scores — "{query_analise}"', y=1.01)
plt.tight_layout()
plt.show()

print(f'Docs com score > 0 : {(scores_all > 0).sum():,}')
print(f'Score máximo       : {scores_all.max():.4f}')
print(f'Score top-10 mín.  : {top50_scores[9]:.4f}')

## 7. Sensibilidade aos parâmetros k1 e b
Justificativa para os valores escolhidos — útil para o relatório.

In [17]:
query_param = 'jazz piano'

configs = [
    {'k1': 1.2, 'b': 0.75},  # padrão Elasticsearch
    {'k1': 1.5, 'b': 0.75},  # nosso padrão
    {'k1': 2.0, 'b': 0.75},  # maior saturação
    {'k1': 1.5, 'b': 0.50},  # menor influência de comprimento
    {'k1': 1.5, 'b': 1.00},  # maior influência de comprimento
]

print(f'Query: "{query_param}"\n')
print(f'{"Config":<20} {"Top-1":<40} {"Score"}')
print('-' * 70)

for cfg in configs:
    idx_tmp = MusicIndex.build(df, k1=cfg['k1'], b=cfg['b'], use_stemming=True)
    res = idx_tmp.search(query_param, k=1)
    top1 = res[0] if res else None
    cfg_str = f"k1={cfg['k1']} b={cfg['b']}"
    if top1:
        nome = f"{top1.track_name[:30]} — {top1.artist_name[:15]}"
        print(f'{cfg_str:<20} {nome:<40} {top1.score:.4f}')
    else:
        print(f'{cfg_str:<20} (sem resultado)')

Query: "jazz piano"

Config               Top-1                                    Score
----------------------------------------------------------------------


34 documentos sem tokens após pré-processamento.


k1=1.2 b=0.75        Just Petals — Piano Jazz Pari            13.2665


34 documentos sem tokens após pré-processamento.


k1=1.5 b=0.75        Just Petals — Piano Jazz Pari            13.7125


KeyboardInterrupt: 

## 8. Comparativo: com stemming vs. sem stemming

In [ ]:
queries_teste = ['dancing', 'rock guitars', 'sad love songs', 'electronic beats']

idx_stem   = MusicIndex.build(df, use_stemming=True)
idx_nostem = MusicIndex.build(df, use_stemming=False)

print(f'{"Query":<25} {"Com stemming":<35} {"Sem stemming"}')
print('-' * 90)

for q in queries_teste:
    r_stem   = idx_stem.search(q, k=1)
    r_nostem = idx_nostem.search(q, k=1)

    top_stem   = f"{r_stem[0].track_name[:25]}  ({r_stem[0].score:.3f})"   if r_stem   else '—'
    top_nostem = f"{r_nostem[0].track_name[:25]}  ({r_nostem[0].score:.3f})" if r_nostem else '—'

    print(f'{q:<25} {top_stem:<35} {top_nostem}')

## 9. Exportar scores BM25 para integração com Partes 2 e 3
Gera um CSV com os scores de um conjunto de queries de teste — usado pela Parte 3 (LTR) e Parte 4 (avaliação).

In [18]:
QUERIES_AVALIACAO = [
    'happy dance pop',
    'sad acoustic ballad',
    'energetic rock guitar',
    'jazz piano instrumental',
    'electronic beats club',
    'classical orchestra symphony',
    'hip hop rap beats',
    'love romantic slow',
    'country blues guitar',
    'metal heavy distortion',
]

records = []
for query in QUERIES_AVALIACAO:
    results = idx.search(query, k=20)
    for r in results:
        records.append({
            'query':       query,
            'rank_bm25':   r.rank,
            'score_bm25':  round(r.score, 6),
            'track_id':    r.track_id,
            'track_name':  r.track_name,
            'artist_name': r.artist_name,
            'popularity':  r.popularity,
            'energy':      r.energy,
            'danceability': r.danceability,
        })

df_bm25_results = pd.DataFrame(records)
output_path = ROOT / 'data' / 'bm25_results_avaliacao.csv'
df_bm25_results.to_csv(output_path, index=False)

print(f'Exportado: {output_path}')
print(f'Queries  : {df_bm25_results["query"].nunique()}')
print(f'Resultados totais: {len(df_bm25_results)}')
df_bm25_results.head(10)

Exportado: C:\Users\raquel\Desktop\Teste\music-search-engine\data\bm25_results_avaliacao.csv
Queries  : 10
Resultados totais: 200


,query,rank_bm25,score_bm25,track_id,track_name,artist_name,popularity,energy,danceability
0,happy dance pop,1,16.454658,0k1rBoFbEGVTyZd1GVbEW5,Happy Dance,Elmo,64.0,0.690,0.931
1,happy dance pop,2,16.454658,335tfwtFty7M3Ge3T6HjbM,Happiness,Dance Gavin Dance,0.0,0.682,0.316
2,happy dance pop,3,15.590245,4wmN0R7EnYR5JPpZPkKFWV,Happy Dance,Bubbles and Friends,22.0,0.825,0.803
3,happy dance pop,4,13.976418,469RsVygmeb4b4uDiSeuao,NASA,Dance Gavin Dance,0.0,0.889,0.363
4,happy dance pop,5,13.136323,4REZIin9q6UtIgir3TVuD3,Pop Off,Dance Gavin Dance,48.0,0.929,0.588
5,happy dance pop,6,13.107410,2ZULQgyL9b282vQwHEvVFU,Strawberry Swisher Pt 2,Dance Gavin Dance,0.0,0.900,0.382
6,happy dance pop,7,13.107410,6A8Wbjp5vNel5wO7mj1rfq,Don t Tell Dave,Dance Gavin Dance,0.0,0.934,0.292
7,happy dance pop,8,13.107410,0C9d07g8OTaJeZVVmjN0pS,Tree Village,Dance Gavin Dance,0.0,0.890,0.170
8,happy dance pop,9,13.107410,07QERwBDKmUPDaQfseaCAR,Powder to the People,Dance Gavin Dance,0.0,0.934,0.249
9,happy dance pop,10,13.107410,41gTdox7kzBEcT7PIz9bHH,Strawberry Swisher Pt 1,Dance Gavin Dance,0.0,0.841,0.280


---
## Resumo dos entregáveis desta etapa

| Arquivo | Destino | Usado por |
|---|---|---|
| `data/bm25_index.pkl` | disco | Partes 3 e 4 |
| `data/bm25_results_avaliacao.csv` | disco | Partes 3 e 4 |
| `src/music_search/indexer.py` | repositório | todos |
| `src/music_search/preprocessing.py` | repositório | todos |